In [ ]:
# atss, coco, plantdoc, epoch 48
# !python tools/train.py configs/atss/atss_r50_fpn_4x_coco-plantdoc.py
!bash tools/dist_train.sh configs/atss/atss_r50_fpn_4x_coco-plantdoc.py 2
!featurize instance release $UUID

/environment/miniconda3/lib/python3.7/site-packages/torch/distributed/launch.py:186: FutureWarning: The module torch.distributed.launch is deprecated
and will be removed in future. Use torchrun.
Note that --use_env is set by default in torchrun.
If your script expects `--local_rank` argument to be set, please
change it to read from `os.environ['LOCAL_RANK']` instead. See 
https://pytorch.org/docs/stable/distributed.html#launch-utility for 
further instructions

  FutureWarning,
*****************************************
Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
*****************************************
/home/featurize/work/.local/lib/python3.7/site-packages/mmengine/utils/dl_utils/setup_env.py:57: UserWarning: Setting MKL_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being over

In [1]:
# test
!python tools/test.py configs/atss/atss_r50_fpn_4x_coco-plantdoc.py \
work_dirs/atss_r50_fpn_4x_coco-plantdoc/best_coco_bbox_mAP_epoch_39.pth

06/17 14:27:29 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.7.10 (default, Jun  4 2021, 14:48:32) [GCC 7.5.0]
    CUDA available: True
    numpy_random_seed: 1737546489
    GPU 0: NVIDIA GeForce RTX 3090
    CUDA_HOME: /usr/local/cuda
    NVCC: Cuda compilation tools, release 11.2, V11.2.152
    GCC: gcc (Ubuntu 9.3.0-17ubuntu1~20.04) 9.3.0
    PyTorch: 1.10.0+cu113
    PyTorch compiling details: PyTorch built with:
  - GCC 7.3
  - C++ Version: 201402
  - Intel(R) Math Kernel Library Version 2020.0.0 Product Build 20191122 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v2.2.3 (Git Hash 7336ca9f055cf1bfa13efb658fe15dc9b41f0740)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: AVX2
  - CUDA Runtime 11.3
  - NVCC architecture flags: -gencode;arch=compute_37,code=sm_37;-gencode;arch=compute_50,cod

In [25]:
# 计算参数量和FLOPS
!python tools/analysis_tools/get_flops.py configs/atss/atss_r50_fpn_4x_coco-plantdoc.py

07/07 15:55:50 - mmengine - WARNING - Some config files, such as configs/yolact and configs/detectors,may have compatibility issues with torch.jit when torch<1.12. If you want to calculate flops for these models, please make sure your pytorch version is >=1.12.
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
/home/featurize/work/mmdetection/mmdet/models/dense_heads/anchor_head.py:108: UserWarning: DeprecationWarning: `num_anchors` is deprecated, for consistency or also use `num_base_priors` instead
  warnings.warn('DeprecationWarning: `num_anchors` is deprecated, '
/environment/miniconda3/lib/python3.7/site-packages/torch/nn/functional.py:2359: UserWarning: __floordiv__ is deprecated, and its behavior will change in a future version of pytorch. It currently rounds toward 0 (like the 'trunc' function NOT 'floor'). This results in incorrect rounding for negative values. To keep the current behavior, use torch.div(a, b, rounding_mode='trunc'), or for act

In [6]:
# calculate Params and FLOPS
import torch
from mmdet.apis import init_detector
from fvcore.nn import FlopCountAnalysis

config_path = "configs/atss/atss_r50_fpn_4x_coco-plantdoc.py"
checkpoint_path = "work_dirs/atss_r50_fpn_4x_coco-plantdoc/best_coco_bbox_mAP_epoch_39.pth"
input_shape = (3, 640, 640)
input_h, input_w = 640, 640
device="cuda:0"

model = init_detector(config_path, checkpoint_path, device=device)
model.eval()

# params
def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

total, trainable = count_parameters(model)

#  FLOPs
dummy_img = torch.randn(1, 3, input_h, input_w).cuda()
dummy_metas = [{"img_shape": (input_h, input_w, 3), "scale_factor": 1.0}]
inputs = (dummy_img, dummy_metas)

flop_analyzer = FlopCountAnalysis(model, inputs)
flops = flop_analyzer.total()   

print("="*60)
print(f"Input shape: 3 × {input_h} × {input_w}")
print(f"Total FLOPs: {flops/1e9:.3f}G")
print(f"Total Params: {total/1e6:.3f}M, Trainable: {trainable/1e6:.3f}M")
print("="*60) 

Loads checkpoint by local backend from path: work_dirs/atss_r50_fpn_4x_coco-plantdoc/best_coco_bbox_mAP_epoch_39.pth


Unsupported operator aten::max_pool2d encountered 1 time(s)
Unsupported operator aten::add_ encountered 16 time(s)
Unsupported operator aten::add encountered 2 time(s)
Unsupported operator aten::mul encountered 5 time(s)
The following submodules of the model were never called during the trace of the graph. They may be unused, or they were accessed by direct calls to .forward() or via other python methods. In the latter case they will have zeros for statistics, though their statistics will still contribute to their parent calling module.
bbox_head.loss_bbox, bbox_head.loss_centerness, bbox_head.loss_cls, bbox_head.relu, data_preprocessor


Input shape: 3 × 640 × 640
Total FLOPs: 81.025G
Total Params: 32.178M, Trainable: 31.952M
